<a href="https://colab.research.google.com/github/Edomario082909/AHHHHR/blob/main/sdxl_v1.0_comfyui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import requests

# 🔑 TOKEN CIVITAI
MY_CIVITAI_TOKEN = "a84327c4d857915fd6730f0f0b637aac"

# =========================
# 🔧 SETUP SISTEMA
# =========================
!apt -y update -qq && apt -y install -qq aria2

# FIX: Attempt to fix libtcmalloc_minimal.so.4 being 'file too short'
LIBTCMALLOC_PATH = "/content/libtcmalloc_minimal.so.4"
!rm -f {LIBTCMALLOC_PATH} # Remove existing potentially corrupt file
!wget -q https://github.com/camenduru/gperftools/releases/download/v1.0/libtcmalloc_minimal.so.4 -O {LIBTCMALLOC_PATH}
# Verify if the downloaded file is not too short (e.g., at least 100KB for a shared library)
import os
if os.path.exists(LIBTCMALLOC_PATH) and os.path.getsize(LIBTCMALLOC_PATH) < 100 * 1024: # Adjust size threshold if known
    print(f"WARNING: {LIBTCMALLOC_PATH} è troppo piccolo dopo il download. Riprovare o indagare la fonte.")
%env LD_PRELOAD={LIBTCMALLOC_PATH}

# =========================
# 📦 INSTALLAZIONE COMFYUI
# =========================
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI

# Prima dipendenze base
!pip install -q torchsde omegaconf diffusers accelerate
!pip install -q -r requirements.txt

# FIX torch / xformers / comfy_kitchen
!pip uninstall -y -q torch torchvision torchaudio xformers
!pip install -q torch==2.10.0 torchvision --index-url https://download.pytorch.org/whl/cu128 # Install torch and torchvision first
!pip install -q torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu128 # Install compatible torchaudio
!pip install -q xformers

# 🔥 FIX crash Ideogram (IMPORTANTE)
!rm -rf /content/ComfyUI/comfy_api_nodes

# =========================
# 🧩 CUSTOM NODES
# =========================
%cd /content/ComfyUI/custom_nodes

!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
!git clone https://github.com/city96/ComfyUI-GGUF.git
!pip install -q gguf
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git

# =========================
# 📥 DOWNLOAD MODELLI (API FIXATA)
# =========================
%cd /content/ComfyUI

!mkdir -p /content/ComfyUI/models/loras
!mkdir -p /content/ComfyUI/models/checkpoints

my_list = [
    ("2747549", "ltx_lora_fov.safetensors", "loras"),
    ("2754108", "ltx_lora_2.safetensors", "loras"),
    ("2617751", "zimage_lora_1.safetensors", "loras"),
    ("2442439", "zimage_checkpoint.safetensors", "checkpoints"),
    ("2465980", "zimage_lora_2.safetensors", "loras"),
    ("2474435", "zimage_lora_3.safetensors", "loras"),
    ("2083303", "wan_lora_1.safetensors", "loras"),
    ("2200388", "wan_lora_2.safetensors", "loras"),
    ("2475163", "zimage_lora_4.safetensors", "loras"),
    ("2477457", "zimage_lora_5.safetensors", "loras"),
    ("2581135", "zimage_lora_6.safetensors", "loras"),
    ("2273468", "wan_lora_3.safetensors", "loras"),
    ("2441730", "wan_lora_4.safetensors", "loras"),
    ("2553151", "wan_lora_5.safetensors", "loras"),
    ("2460437", "zimage_lora_7.safetensors", "loras"),
    ("2073605", "wan_lora_6.safetensors", "loras"),
    ("2739037", "sdxl_real_checkpoint.safetensors", "checkpoints"),
    ("1811054", "sdxl_lora_1.safetensors", "loras"),
    ("2752410", "ltx_checkpoint.safetensors", "checkpoints"),
    ("2376136", "wan_lora_7.safetensors", "loras"),
    ("2342652", "wan_lora_8.safetensors", "loras"),
    ("2176450", "wan_lora_9.safetensors", "loras"),
    ("2430424", "wan_lora_10.safetensors", "loras"),
    ("2738377", "grok_checkpoint.safetensors", "checkpoints")
]

def download_model(v_id, name, folder):
    url = f"https://civitai.com/api/v1/model-versions/{v_id}"
    headers = {"Authorization": f"Bearer {MY_CIVITAI_TOKEN}"}

    try:
        r = requests.get(url, headers=headers, timeout=30)
        r.raise_for_status()
        data = r.json()

        files = data.get("files", [])
        if not files or "downloadUrl" not in files[0]:
            raise ValueError("downloadUrl non trovato nella risposta API")

        file_url = files[0]["downloadUrl"]

        print(f"⬇️ {name}")

        os.system(f'''
        aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
        "{file_url}" \
        -d /content/ComfyUI/models/{folder} \
        -o {name}
        ''')

        print(f"✅ OK: {name}")

    except Exception as e:
        print(f"❌ Skip: {name} ({e})")

for v_id, name, folder in my_list:
    download_model(v_id, name, folder)

# =========================
# 🌐 CLOUDFARE TUNNEL
# =========================
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared-linux-amd64
!chmod 777 /content/cloudflared-linux-amd64

import threading, subprocess, re

def tunnel():
    p = subprocess.Popen(
        ['/content/cloudflared-linux-amd64', 'tunnel', '--url', 'http://127.0.0.1:8188'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    for line in iter(p.stdout.readline, b''):
        line = line.decode(errors="ignore")
        link = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if link:
            print(f"\n🟢 LINK COMFYUI: {link.group(0)}\n")

threading.Thread(target=tunnel, daemon=True).start()

# =========================
# ▶️ AVVIO COMFYUI
# =========================
!python main.py --dont-print-server --listen 0.0.0.0

In [ ]:
# 1. Installa la libreria mancante per i modelli GGUF
!pip install gguf

# 2. Forza l'installazione di WanVideo che era fallita
# %cd /content/ComfyUI/custom_nodes # Commentato
# !rm -rf ComfyUI-WanVideo # Rimuove la cartella se vuota o corrotta # Commentato
# !git clone https://github.com/kijai/ComfyUI-WanVideo.git # Commentato
# %cd ComfyUI-WanVideo # Commentato
# !pip install -r requirements.txt # Commentato

# 3. Torna alla cartella principale
%cd /content/ComfyUI

print("\n✅ Riparazione completata! Ora riavvia la cella di ComfyUI.")